# APIs Lab

In this lab, you will work with different APIs to build datasets and perform basic analysis.

The goal is to practice how to:

- Make API requests  
- Explore JSON responses  
- Extract relevant information  
- Transform data into pandas DataFrames  
- Generate simple insights  

You will work like a data analyst using APIs data.

# Challenge 1: Pokémon Dataset & Analysis

## Goal

Build and analyze a Pokémon dataset using the API.

## Example (single request)

Below you can see how to get data for ONE Pokémon:

In [ ]:
import requests

url = "https://pokeapi.co/api/v2/pokemon/1"

response = requests.get(url)
data = response.json()

data

## Tasks

1. Using the example above, get data for Pokémon IDs from 1 to 151

2. Extract:
   - name
   - height
   - weight
   - base_experience
   - first type

3. Create a DataFrame

4. Add a new column:
   - weight_kg (divide weight by 10)

5. Answer:
   - Which Pokémon is the heaviest?
   - What is the average weight?
   - How many Pokémon are there per type?

6. Answer:
   - Which type has the highest average weight?

In [ ]:
import requests

base_url = "https://pokeapi.co/api/v2/pokemon/"
pokemon_ids = range(1, 51)
pokemon_data = []

for pokemon_id in pokemon_ids:
    url = base_url + str(pokemon_id)
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        pokemon_data.append(data)
    else:
        print("Failed to fetch data for Pokemon ID", pokemon_id)

In [ ]:
for data in pokemon_data:
    name = data['name']
    height= data['height']
    weight= data['weight']
    base_experience= data['base_experience']
    first_type= data['types'][0]
    print(name + " - " + str(height) + " - " + str(weight) + " - " + str(base_experience) + " - " + str(first_type))

In [ ]:

import pandas as pd

df = pd.DataFrame(pokemon_data)

In [ ]:
#Step 4
def convert_weight(weight):
    return weight / 10

df['weight_kg'] = df['weight'].apply(convert_weight)

In [ ]:
heaviest_idx = df['weight_kg'].idxmax()
heaviest_pokemon = df.loc[heaviest_idx, 'name']
heaviest_weight = df.loc[heaviest_idx, 'weight_kg']
print(heaviest_weight)

In [ ]:
average_weight = df['weight_kg'].mean()
print(average_weight)

In [ ]:
df['weight_kg'].value_counts()

In [ ]:
df['types_list'] = df['types'].apply(lambda x: [t['type']['name'] for t in x])
df_exploded = df.explode('types_list')

avg_weight_per_type = df_exploded.groupby('types_list')['weight_kg'].mean()
heaviest_type = avg_weight_per_type.idxmax()
heaviest_type_val = avg_weight_per_type.max()
print(heaviest_type_val)

# Challenge 2: Crypto Market Analysis

## Goal

Analyze cryptocurrency prices using real-time data.

## Tasks

1. Get prices for:
   - bitcoin
   - ethereum
   - solana
   - dogecoin

2. Convert the response into a DataFrame (coins as rows)

3. Rename columns properly

4. Add:
   - price_rank (ranking by price)
   - price_category:
        - "high" if > 1000  
        - "medium" if between 100 and 1000  
        - "low" if < 100  

5. Answer:
   - Which coin is the most expensive?
   - Which category has more coins?

6. **Bonus**:
   - Create a column with price difference vs bitcoin

In [89]:
import requests
import pandas as pd

url = "https://api.coingecko.com/api/v3/simple/price"

params = {
    "ids": "bitcoin,ethereum,solana,dogecoin",
    "vs_currencies": "usd"
}

In [90]:
# Your code goes here
response = requests.get(url ,headers={"Accept": "application/json"}, params=params)
currency_data = response.json()
currency_data

{'bitcoin': {'usd': 70870},
 'dogecoin': {'usd': 0.090857},
 'ethereum': {'usd': 2186.24},
 'solana': {'usd': 81.88}}

In [91]:
currency_df = pd.DataFrame(currency_data)

currency_df = currency_df.T
currency_df = currency_df.rename(columns={'usd': 'Price_In_USD'})
currency_df = currency_df.reset_index().rename(columns={'index': 'Coin_Name'})
currency_df

,Coin_Name,Price_In_USD
0,bitcoin,70870.000000
1,dogecoin,0.090857
2,ethereum,2186.240000
3,solana,81.880000


In [92]:
currency_df['Price_Rank'] = currency_df['Price_In_USD'].rank(method='first', ascending=False)
currency_df.sort_values(by='Price_Rank', inplace=True)

currency_df

,Coin_Name,Price_In_USD,Price_Rank
0,bitcoin,70870.000000,1.0
2,ethereum,2186.240000,2.0
3,solana,81.880000,3.0
1,dogecoin,0.090857,4.0


In [95]:
currency_df['Price_Category']=currency_df['Price_In_USD'].apply(lambda x: 'high' if x>1000 else ('medium' if x>100 else 'low'))
currency_df

,Coin_Name,Price_In_USD,Price_Rank,Price_Category
0,bitcoin,70870.000000,1.0,high
2,ethereum,2186.240000,2.0,high
3,solana,81.880000,3.0,low
1,dogecoin,0.090857,4.0,low


In [94]:
float(currency_df['Price_In_USD'][0])

70870.0

In [96]:
currency_df['Coin_Name'][0]

'bitcoin'